# Notebook 04 - Regression, Price per Square Meter, and Location

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
df = pd.read_csv(PROJECT_ROOT / "data/processed/mexico-real-combined-clean.csv")
df["price_per_m2"] = df["price_usd"] / df["area_m2"]

Total price and price per square meter answer different questions. Price per square meter is total price divided by area, so comparisons with area are mathematically coupled.

In [ ]:
MIN_STATE_N = 10
state_stats = df.groupby("state").agg(
    sample_size=("price_usd", "size"),
    mean_total_price=("price_usd", "mean"),
    median_total_price=("price_usd", "median"),
    mean_price_per_m2=("price_per_m2", "mean"),
    median_price_per_m2=("price_per_m2", "median"),
).sort_values("median_total_price", ascending=False)
eligible = state_stats[state_stats["sample_size"] >= MIN_STATE_N].copy()
excluded = state_stats[state_stats["sample_size"] < MIN_STATE_N].copy()
print("State-level table:")
display(state_stats.round(2))
print("States ranked by median total price (n >= 10):")
display(eligible.round(2))
print("States excluded from rankings:")
display(excluded.round(2))

The minimum-10 rule applies only to comparative rankings. Excluded observations remain in the full dataset and all regression analyses.

In [ ]:
model = smf.ols("np.log(price_usd) ~ area_m2 + C(state) + C(property_type)", data=df).fit(cov_type="HC3")
summary_table = pd.DataFrame({
    "coefficient": model.params,
    "std_error": model.bse,
    "ci_low": model.conf_int()[0],
    "ci_high": model.conf_int()[1],
    "p_value": model.pvalues,
})
print("Observations:", int(model.nobs))
print("R-squared:", round(model.rsquared, 4))
print("Adjusted R-squared:", round(model.rsquared_adj, 4))
display(summary_table.round(6))

In [ ]:
area_coef = model.params["area_m2"]
area_pct = (np.exp(area_coef) - 1) * 100
print(f"Area coefficient: {area_coef:.6f} log USD per m2")
print(f"Approximate multiplicative price change per 1 m2: {area_pct:.3f}%")
print("This is an adjusted association, not a causal effect.")

In [ ]:
resid = model.resid
fitted = model.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, resid, alpha=0.25, s=12)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set(title="Residuals vs. Fitted", xlabel="Fitted log price", ylabel="Residual")
sm.qqplot(resid, line="45", ax=axes[1])
axes[1].set_title("Residual Q-Q Plot")
plt.tight_layout()
plt.show()
bp = het_breuschpagan(resid, model.model.exog)
print("Breusch-Pagan LM statistic:", round(bp[2], 4))
print("Breusch-Pagan p-value:", round(bp[3], 6))

In [ ]:
design = pd.get_dummies(df[["area_m2", "state", "property_type"]], columns=["state", "property_type"], drop_first=True, dtype=float)
design = sm.add_constant(design)
vifs = pd.DataFrame({"feature": design.columns, "VIF": [variance_inflation_factor(design.values, i) for i in range(design.shape[1])]})
print(vifs[vifs.feature != "const"].sort_values("VIF", ascending=False).head(15).round(3).to_string(index=False))

In [ ]:
model_a = smf.ols("np.log(price_usd) ~ C(state)", data=df).fit(cov_type="HC3")
model_b = smf.ols("np.log(price_usd) ~ area_m2 + C(state)", data=df).fit(cov_type="HC3")
model_c = smf.ols("np.log(price_usd) ~ area_m2 + C(state) + C(property_type)", data=df).fit(cov_type="HC3")
model_d = smf.ols("np.log(price_usd) ~ C(state) + C(property_type)", data=df).fit(cov_type="HC3")
comparison = pd.DataFrame({
    "model": ["A: state", "B: area + state", "C: area + state + property type", "D: state + property type"],
    "n": [int(model_a.nobs), int(model_b.nobs), int(model_c.nobs), int(model_d.nobs)],
    "R_squared": [model_a.rsquared, model_b.rsquared, model_c.rsquared, model_d.rsquared],
    "adjusted_R_squared": [model_a.rsquared_adj, model_b.rsquared_adj, model_c.rsquared_adj, model_d.rsquared_adj],
})
comparison["change_vs_parent"] = [float("nan"), model_b.rsquared-model_a.rsquared,
                                   model_c.rsquared-model_b.rsquared,
                                   model_d.rsquared-model_a.rsquared]
display(comparison.round(6))
print(f"Area addition to state model: {model_b.rsquared-model_a.rsquared:.6f} R-squared")
print(f"Property-type addition to area + state model: {model_c.rsquared-model_b.rsquared:.6f} R-squared")
print(f"Property-type addition to state model: {model_d.rsquared-model_a.rsquared:.6f} R-squared")
print(f"Area addition to state + property-type model: {model_c.rsquared-model_d.rsquared:.6f} R-squared")

In [ ]:
df_without_dataset2 = df[df["source_dataset"] != "mexico-real-estate-2.csv"].copy()
model_without_dataset2 = smf.ols("np.log(price_usd) ~ area_m2 + C(state) + C(property_type)", data=df_without_dataset2).fit(cov_type="HC3")
sensitivity = pd.DataFrame({
    "sample": ["Full cleaned data", "Dataset 2 excluded"],
    "n": [int(model.nobs), int(model_without_dataset2.nobs)],
    "area_coefficient": [model.params["area_m2"], model_without_dataset2.params["area_m2"]],
    "R_squared": [model.rsquared, model_without_dataset2.rsquared],
    "adjusted_R_squared": [model.rsquared_adj, model_without_dataset2.rsquared_adj],
})
display(sensitivity.round(6))

## Model interpretation and limitations

The nested-model R-squared comparisons use the same 2,082 observations and are model-based explanatory comparisons, not causal effects. R-squared does not prove that one factor is universally more important. Area is associated with total listing price; state captures only broad state-level differences. The main regression does not incorporate neighborhood-level or more detailed spatial information. State should therefore not be interpreted as a complete location measure.